# CalibMiss data exploration

Explore the frozen OHLCV universe, engineered predictors, simulated missingness, and a chronological logistic-regression reference. This is diagnostic analysis, not a trading strategy.

Run `make data` and `make features` from the repository root before running notebook.

## Setup and configuration

Use project functions and cached files only. The notebook does not redownload data or modify raw files.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd() if (Path.cwd() / 'config').is_dir() else Path.cwd().parent
if not (ROOT / 'config').is_dir():
    raise FileNotFoundError('Run this notebook from the repository root or notebooks directory.')
sys.path.insert(0, str(ROOT / 'src'))

from calibmiss.config import get_config, get_mar_rules
from calibmiss.data import validate_ohlcv
from calibmiss.imputers import get_imputer
from calibmiss.metrics import compute_metrics
from calibmiss.missingness import MARRuleConfig, inject_mar, inject_mcar
from calibmiss.models import get_model
from calibmiss.splits import get_split_indices

sns.set_theme(style='whitegrid', context='notebook')
cfg = get_config(ROOT / 'config')
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
raw_files = sorted(RAW_DIR.glob('*.parquet')) if RAW_DIR.exists() else []
features_path = PROCESSED_DIR / 'features.parquet'
target_path = PROCESSED_DIR / 'target.parquet'
if not raw_files or not features_path.exists() or not target_path.exists():
    raise FileNotFoundError('Cached raw/features/target data missing; run `make data` and `make features`.')
raw = pd.read_parquet(raw_files[-1])
X = pd.read_parquet(features_path)
y = pd.read_parquet(target_path)['target']
X, y = X.align(y, join='inner', axis=0)
print(f'{len(cfg.universe.tickers)} configured tickers, {len(raw):,} raw rows, {len(X):,} feature rows')
print(f"Date range: {X.index.get_level_values('date').min().date()} to {X.index.get_level_values('date').max().date()}")

## Data quality

Report source-data validation and native missingness separately from experiment-injected missingness.

In [ ]:
quality = validate_ohlcv(raw)
print(f'Valid OHLCV: {quality.is_valid}')
if quality.errors:
    display(pd.Series(quality.errors, name='validation_error').to_frame())
display(pd.Series(quality.native_missing_rates, name='native_missing_rate').to_frame())
ticker_counts = raw.groupby(level='ticker').size().rename('rows')
display(ticker_counts.describe().to_frame().T)
display(ticker_counts.to_frame())
print(f'Target up-day rate: {y.mean():.3f}; target rows: {len(y):,}')

## Feature distributions

Inspect pooled feature ranges and distributions. Extreme tails may affect standardized linear models; rows remain ticker-date observations, not independent statistical replicates.

In [ ]:
display(X.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T)
axes = X.hist(bins=40, figsize=(14, 10), color='#2878a0', edgecolor='white')
for axis in axes.flat:
    axis.set_ylabel('Count')
plt.tight_layout()
plt.show()

## Missingness

Compare native missing rates with one deterministic MCAR and configured MAR sample. Samples are diagnostic only; full experiments use their complete configured grid and seeds.

In [ ]:
sample_rate = 0.3
sample_seed = cfg.experiment.missingness.seeds[0]
mcar_X, mcar_rates = inject_mcar(X, sample_rate, sample_seed)
mar_config = get_mar_rules(ROOT / 'config').mar_rules[0]
mar_rule = MARRuleConfig(**mar_config.model_dump())
mar_X, mar_rates = inject_mar(X, mar_rule, sample_rate, sample_seed)
missing_rates = pd.DataFrame({
    'native OHLCV (mean)': pd.Series(quality.native_missing_rates),
    'MCAR feature sample': pd.Series(mcar_rates),
    'MAR target feature': pd.Series(mar_rates),
}).fillna(0)
display(missing_rates)
missing_rates.plot(kind='bar', figsize=(12, 4), color=['#64748b', '#2878a0', '#d97706'])
plt.ylabel('Missing fraction')
plt.xlabel('')
plt.xticks(rotation=35, ha='right')
plt.legend(frameon=False)
plt.tight_layout()
plt.show()
print(f'MCAR target {sample_rate:.0%}; realized mean {mcar_X.isna().to_numpy().mean():.3f}')
print(f'MAR target {sample_rate:.0%}; realized {mar_rates[mar_rule.target_feature]:.3f} in {mar_rule.target_feature}')

## Baseline model

Fit the configured logistic-regression baseline on the first chronological walk-forward split. Split boundaries are created from unique dates with configured purge gaps, then mapped to all ticker rows for those dates. Imputer/model fit only on training observations; evaluate Brier, ECE, and ROC AUC on held-out test rows. This diagnostic is one fold and seed, not the complete grid.

In [ ]:
dates = pd.DatetimeIndex(X.index.get_level_values('date').unique()).sort_values()
split_cfg = cfg.experiment.splits.model_dump()
date_splits = get_split_indices(dates, split_cfg)
if not date_splits:
    raise ValueError('Configured date range does not contain a complete walk-forward split.')
split = date_splits[0]
row_dates = pd.DatetimeIndex(X.index.get_level_values('date'))
date_positions = {date: position for position, date in enumerate(dates)}
row_date_positions = np.fromiter((date_positions[date] for date in row_dates), dtype=int, count=len(X))
train_idx, _cal_idx, test_idx = [
    np.flatnonzero(np.isin(row_date_positions, date_idx))
    for date_idx in (split.train_idx, split.calibrate_idx, split.test_idx)
]
imputer_cfg = next(item for item in cfg.experiment.imputers if item.name == 'median')
model_cfg = next(item for item in cfg.experiment.models if item.name == 'logistic_regression')
baseline_X_train = X.iloc[train_idx]
baseline_X_test = X.iloc[test_idx]
baseline_y_train = y.iloc[train_idx]
baseline_y_test = y.iloc[test_idx]
imputer = get_imputer(imputer_cfg)
imputer.fit(baseline_X_train)
baseline_X_train = imputer.transform(baseline_X_train)
baseline_X_test = imputer.transform(baseline_X_test)
model = get_model(model_cfg)
model.fit(baseline_X_train, baseline_y_train)
probabilities = model.predict_proba(baseline_X_test)
baseline_metrics = compute_metrics(baseline_y_test.to_numpy(), probabilities, cfg.experiment.evaluation.n_bins_ece)
display(pd.Series({key: value for key, value in baseline_metrics.items() if key != 'reliability'}, name='test_metric').to_frame())
naive_probability = np.full(len(baseline_y_test), baseline_y_train.mean())
naive_metrics = compute_metrics(baseline_y_test.to_numpy(), naive_probability, cfg.experiment.evaluation.n_bins_ece)
display(pd.DataFrame([
    {'baseline': 'logistic_regression_median', **{key: value for key, value in baseline_metrics.items() if key != 'reliability'}},
    {'baseline': 'naive_training_base_rate', **{key: value for key, value in naive_metrics.items() if key != 'reliability'}},
]))
print(f'Train {split.train_start.date()}–{split.train_end.date()} ({len(train_idx):,} rows); calibration {split.calibrate_start.date()}–{split.calibrate_end.date()}; test {split.test_start.date()}–{split.test_end.date()} ({len(test_idx):,} rows)')

## Reliability

Calibration is central to this study; inspect test-fold reliability bins alongside scalar metrics. Empty bins are omitted.

In [ ]:
reliability = baseline_metrics['reliability']
counts = np.asarray(reliability['bin_counts'])
occupied = counts > 0
confidence = np.asarray(reliability['bin_confidences'])[occupied]
accuracy = np.asarray(reliability['bin_accuracies'])[occupied]
plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], color='#64748b', linestyle='--', linewidth=1.5, label='Perfect calibration')
plt.scatter(confidence, accuracy, s=48, color='#2878a0', edgecolor='white', linewidth=0.8, label='Test bins')
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.xlabel('Mean predicted probability')
plt.ylabel('Observed up-day fraction')
plt.legend(frameon=False)
plt.tight_layout()
plt.show()
display(pd.DataFrame({'count': counts[occupied], 'mean_probability': confidence, 'observed_fraction': accuracy}))

## Interpretation notes

- Native OHLCV missingness is a source-quality diagnostic; controlled experiments use separately injected feature missingness.
- Feature histograms pool tickers and dates; dependence across rows means row count is not an independent sample size.
- This baseline uses a single walk-forward test window and no post-hoc calibrator. Compare definitive results using `results/metrics.csv`, fold/seed summaries, and paired tests.
- Do not infer trading profitability from calibration or discrimination metrics.